In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix, roc_curve)

plt.style.use('seaborn-v0_8')
RANDOM_STATE = 42


def train_disease(name, target_col, csv_path, apply_smote=False):
    """Reusable training pipeline for one disease."""
    print(f"\n{'='*60}")
    print(f"TRAINING: {name.upper()}")
    print(f"{'='*60}")

    # 1. Load
    df = pd.read_csv(csv_path)
    print(f"Loaded: {df.shape}")

    # 2. Split features/target
    X = df.drop(columns=[target_col])
    y = df[target_col]

    # 3. Split BEFORE imputation (prevents leakage)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    print(f"Train: {X_train.shape} | Test: {X_test.shape}")
    print(f"Train pos rate: {y_train.mean():.2%} | Test pos rate: {y_test.mean():.2%}")

    # 4. Impute (fit on train only)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = pd.DataFrame(
        imputer.fit_transform(X_train), columns=X_train.columns
    )
    X_test_imp = pd.DataFrame(
        imputer.transform(X_test), columns=X_test.columns
    )

    # 5. SMOTE (only on train!)
    if apply_smote:
        smote = SMOTE(random_state=RANDOM_STATE)
        X_train_imp, y_train = smote.fit_resample(X_train_imp, y_train)
        print(f"After SMOTE — train shape: {X_train_imp.shape}, pos rate: {y_train.mean():.2%}")

    # 6. Scale (fit on train only)
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train_imp), columns=X_train.columns
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test_imp), columns=X_test.columns
    )

    # 7. Models
    models = {
        'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, class_weight='balanced'),
        'Random Forest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'),
        'XGBoost': XGBClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss', verbosity=0),
        'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
        'KNN (k=5)': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    }

    # 8. Train + evaluate
    results = []
    trained = {}
    for mname, model in models.items():
        t0 = time.time()
        model.fit(X_train_scaled, y_train)
        train_time = time.time() - t0

        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]

        results.append({
            'Model': mname,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'F1': f1_score(y_test, y_pred, zero_division=0),
            'ROC-AUC': roc_auc_score(y_test, y_proba),
            'Train Time (s)': round(train_time, 2),
        })
        trained[mname] = model

    results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
    print("\n" + results_df.to_string(index=False))

    # 9. Pick best
    best_name = results_df.iloc[0]['Model']
    best_model = trained[best_name]
    print(f"\n🏆 Best: {best_name}")

    # 10. Save
    joblib.dump(best_model, f'../models/{name}_model.pkl')
    joblib.dump(scaler, f'../models/{name}_scaler.pkl')
    joblib.dump(imputer, f'../models/{name}_imputer.pkl')
    joblib.dump(list(X.columns), f'../models/{name}_features.pkl')
    joblib.dump(best_name, f'../models/{name}_model_name.pkl')
    results_df.to_csv(f'../reports/{name}_comparison.csv', index=False)

    # 11. ROC curves
    plt.figure(figsize=(8, 6))
    for mname, model in trained.items():
        if hasattr(model, 'predict_proba'):
            y_p = model.predict_proba(X_test_scaled)[:, 1]
            fpr, tpr, _ = roc_curve(y_test, y_p)
            auc = roc_auc_score(y_test, y_p)
            plt.plot(fpr, tpr, label=f'{mname} (AUC={auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{name.title()} — ROC Curves')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(f'../reports/{name}_roc_curves.png', dpi=150)
    plt.close()

    # 12. Confusion matrix for best
    y_pred_best = best_model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred_best)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    plt.title(f'{name.title()} — Confusion Matrix ({best_name})')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'../reports/{name}_confusion_matrix.png', dpi=150)
    plt.close()

    return {
        'name': name,
        'best_model': best_name,
        'results_df': results_df,
        'best_metrics': results_df.iloc[0].to_dict()
    }

print("✅ Helper function defined")

✅ Helper function defined


In [2]:
diabetes_result = train_disease(
    name='diabetes',
    target_col='Outcome',
    csv_path='../data/processed/diabetes_clean.csv',
    apply_smote=False    # PIMA is mildly imbalanced — class_weight handles it
)


TRAINING: DIABETES
Loaded: (768, 9)
Train: (614, 8) | Test: (154, 8)
Train pos rate: 34.85% | Test pos rate: 35.06%

              Model  Accuracy  Precision   Recall       F1  ROC-AUC  Train Time (s)
Logistic Regression  0.733766   0.603175 0.703704 0.649573 0.812593            0.02
      Random Forest  0.733766   0.638298 0.555556 0.594059 0.811852            0.58
            XGBoost  0.759740   0.680851 0.592593 0.633663 0.802593            0.34
          SVM (RBF)  0.740260   0.652174 0.555556 0.600000 0.796389            0.13
          KNN (k=5)  0.753247   0.660000 0.611111 0.634615 0.789907            0.01

🏆 Best: Logistic Regression


In [3]:
heart_result = train_disease(
    name='heart',
    target_col='HeartDisease',
    csv_path='../data/processed/heart_clean.csv',
    apply_smote=False    # 55/45 — already balanced
)


TRAINING: HEART
Loaded: (918, 16)
Train: (734, 15) | Test: (184, 15)
Train pos rate: 55.31% | Test pos rate: 55.43%

              Model  Accuracy  Precision   Recall       F1  ROC-AUC  Train Time (s)
          SVM (RBF)  0.853261   0.850467 0.892157 0.870813 0.941416            0.16
          KNN (k=5)  0.891304   0.894231 0.911765 0.902913 0.935497            0.00
      Random Forest  0.875000   0.891089 0.882353 0.886700 0.932508            0.67
Logistic Regression  0.880435   0.900000 0.882353 0.891089 0.931492            0.01
            XGBoost  0.853261   0.894737 0.833333 0.862944 0.909374            0.25

🏆 Best: SVM (RBF)


In [4]:
liver_result = train_disease(
    name='liver',
    target_col='Dataset',
    csv_path='../data/processed/liver_clean.csv',
    apply_smote=True     # 71/29 — significant imbalance
)


TRAINING: LIVER
Loaded: (583, 11)
Train: (466, 10) | Test: (117, 10)
Train pos rate: 71.46% | Test pos rate: 70.94%
After SMOTE — train shape: (666, 10), pos rate: 50.00%

              Model  Accuracy  Precision   Recall       F1  ROC-AUC  Train Time (s)
          SVM (RBF)  0.717949   0.878788 0.698795 0.778523 0.850461            0.16
Logistic Regression  0.717949   0.903226 0.674699 0.772414 0.836286            0.02
      Random Forest  0.726496   0.780220 0.855422 0.816092 0.782247            0.62
            XGBoost  0.692308   0.747368 0.855422 0.797753 0.704111            0.19
          KNN (k=5)  0.589744   0.761194 0.614458 0.680000 0.662296            0.00

🏆 Best: SVM (RBF)
